In [ ]:
import sys
sys.path.append('../..')

import torch
from data.utils import get_datasets
from models.transformer import Transformer
from models.train import train, predict
from models.evaluate import evaluate
from plots import plot_training_curves, plot_confusion_matrix, plot_per_class_metrics

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
train_ds, valid_ds, test_ds = get_datasets('mfcc', train_path='../../data/train', valid_path='../../data/valid', test_path='../../data/test')
print(f'Train: {len(train_ds)}, Valid: {len(valid_ds)}, Test: {len(test_ds)}')

In [ ]:
transformer = Transformer(n_features=40, n_timesteps=101, num_classes=12, d_model=128, nhead=4, num_layers=4, dropout=0.1, pooling='mean').to(device)
total_params = sum(p.numel() for p in transformer.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
model, history = train(transformer, train_ds, valid_ds, epochs=30, batch_size=64, lr=3e-4, device=device, checkpoint_name='transformer_mfcc')

In [ ]:
plot_training_curves(history, title='Transformer – MFCC')

In [ ]:
preds_valid, labels_valid = predict(model, valid_ds, device=device)
result_valid = evaluate(preds_valid, labels_valid)
plot_confusion_matrix(result_valid['cm'], normalize=True, title='Transformer – MFCC (valid)')
plot_per_class_metrics(labels_valid.numpy(), preds_valid.numpy(), title='Transformer – MFCC (valid)')

In [ ]:
preds_test, labels_test = predict(model, test_ds, device=device)
result_test = evaluate(preds_test, labels_test)
plot_confusion_matrix(result_test['cm'], normalize=True, title='Transformer – MFCC (test)')
plot_per_class_metrics(labels_test.numpy(), preds_test.numpy(), title='Transformer – MFCC (test)')